# Campus Regulations Assistant (Problem Statement 2A)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/manyatha-313/student-chatbot/blob/main/notebooks/campus_assistant_2A.ipynb)

An assistant that answers students' questions about **Sri Siddhartha Institute of Technology (SSIT), Tumakuru** using only the official college documents: the **Hand Book cum Work Planner 2025-26** and the **department syllabus** (B.E. CSE, ECE, ME, IT; M.Tech. CSE, VLSI-ES, DE, PDM, TPE).

Built from the Hack Day Theme 2 starter kit. What changed from the starter kit:

| Starter kit | This notebook |
|---|---|
| One sample document, split into 120-word chunks in the notebook | **3,600+ prepared chunks** loaded from `dataset/processed/chunks.jsonl` (split by clause and by course unit, see `plans/datasetplan.md`) |
| - | One non-SSIT document (an **upGrad School of Technology brochure**) is included but marked: the assistant never presents it as an SSIT rule, and `INCLUDE_NON_SSIT = False` leaves it out |
| Embeds the chunk text | Embeds **`embed_text`**: the document, section and heading in front of the text, so short questions find the right clause |
| Cites the page | Cites the **document and page**, e.g. *(SSIT_Handbook_2025-26.pdf, p. 21)* |
| "The answer could not be found" | Declines and **names the office to contact**, taken from the chunk's `contact` field |
| Always calls Gemma | **Relevance threshold:** declines without calling Gemma when no passage is relevant enough |
| One PDF at a time | All documents at once, a **course filter**, and extra PDFs can be uploaded |

```
 chunks.jsonl ──► embed each chunk's embed_text (all-MiniLM-L6-v2)
                                   │
 Question ──► retrieve the TOP_K closest chunks (optionally only the student's course + the handbook)
                                   │
              best similarity < MIN_SIMILARITY ?  ──yes──► decline + contact office (Gemma not called)
                                   │ no
              give ONLY those chunks to Gemma ──► answer with (document, p. N) citations
```

### Set-up (once per team member)
1. **File → Save a copy in Drive**, so your changes are saved.
2. Create a free key at **https://aistudio.google.com/apikey**.
3. In Colab, open **Secrets** (key icon, left sidebar), add `GOOGLE_API_KEY` with your key, and enable **Notebook access**.
4. **Runtime → Run all**. The default CPU runtime is enough. Embedding the chunks takes about 1-2 minutes the first time.

> **Important:** do not paste the key into a code cell.
>
> **No key yet:** the notebook still runs. Retrieval works normally, and the app shows the best passage in place of a written answer.

## Step 0: Installation (run once)

In [ ]:
# Installs the required libraries. This takes about one minute.
!pip install -q pypdf sentence-transformers "gradio>=6,<7" google-genai

## Step 1: Settings

The main values to experiment with. Re-run this cell and every cell below it after changing one.

In [ ]:
# The Gemma 4 model that writes the answers.
#   "gemma-4-26b-a4b-it"  Mixture of Experts, 26B total / about 4B active parameters. Fast. (default)
#   "gemma-4-31b-it"      Dense, 31B parameters. More capable, slower.
GEMMA_MODEL = "gemma-4-26b-a4b-it"

# The prepared chunks (made by scripts/prepare_dataset.py from the college PDFs).
CHUNKS_URL = "https://raw.githubusercontent.com/manyatha-313/student-chatbot/main/dataset/processed/chunks.jsonl"

# The number of chunks given to Gemma for each question.
TOP_K = 4

# Relevance threshold: if the best chunk's similarity is below this, the assistant declines without calling Gemma.
#   Higher = declines more often (safer); lower = tries to answer more (riskier). Tune it on the test set.
MIN_SIMILARITY = 0.30

# The dataset also holds one document that is not from SSIT (an upGrad School of Technology brochure).
#   True  = search it too; answers that use it say it is not an SSIT document.
#   False = leave it out, so answers come only from SSIT / SSAHE documents.
INCLUDE_NON_SSIT = True

# Whom to suggest when nothing relevant is found at all (e.g. a question that is not about the college).
DEFAULT_CONTACT = "the Office of the Dean (Academics), SSIT"

# Chunking for PDFs uploaded in the app (the prepared chunks are already split).
CHUNK_SIZE = 120
CHUNK_OVERLAP = 30

# The words the assistant uses when it declines. Gemma is told to use exactly this sentence.
DECLINE = "I could not find this in the official college documents. Please contact {contact}."

# The rules Gemma follows when answering.
ANSWER_RULES = f"""You are the Campus Regulations Assistant of Sri Siddhartha Institute of Technology (SSIT), Tumakuru.
You answer students' questions using ONLY the context passages provided. Never use outside knowledge and never guess.
- Each passage begins with its source in brackets, for example [SSIT_Handbook_2025-26.pdf, Page 21, SRE 7.1], followed by the office to contact.
- After each fact you use, cite its document and page in brackets, for example (SSIT_Handbook_2025-26.pdf, p. 21).
- If passages for different programmes or documents give different rules (for example B.E. and MBA), state each rule and say which programme it applies to.
- If the answer is not in the context, reply exactly: "{DECLINE}" and replace {{contact}} with the contact office of the most relevant passage.
- A passage marked NOT AN SSIT DOCUMENT comes from another institution. Never present it as an SSIT rule. If you use it, say "according to the upGrad School of Technology brochure (not an SSIT document)".
- Keep answers concise: 2 to 5 sentences."""

TEMPERATURE = 0.2       # a low value keeps answers consistent and close to the documents

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # optional experiment: "BAAI/bge-small-en-v1.5"
print("Settings configured.")

## Step 2: Connect to Gemma and load the embedding model

In [ ]:
import os
from google import genai
from google.genai import types, errors


def _get_secret(name):
    """Reads a secret from the Colab Secrets panel (or from an environment variable)."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)


GOOGLE_API_KEY = _get_secret("GOOGLE_API_KEY")
DEMO_MODE = not GOOGLE_API_KEY
client = None if DEMO_MODE else genai.Client(api_key=GOOGLE_API_KEY)


def _friendly_error(e):
    code = getattr(e, "code", None)
    if code in (400, 401, 403) and "key" in str(e).lower():
        return "Error: the GOOGLE_API_KEY was rejected. Check that the complete key is saved in Colab Secrets, then re-run this cell."
    if code == 429:
        return "Error: the free-tier usage limit has been reached. Wait about one minute and try again."
    if code == 404:
        return f"Error: the model '{GEMMA_MODEL}' was not found. Check the GEMMA_MODEL setting."
    return f"Error: the request to Gemma failed ({e})."


def generate(user_text, system_text=None, temperature=TEMPERATURE, demo_reply="(Demonstration mode response)"):
    """Sends one message to Gemma and returns its reply as text.
    In demonstration mode, returns `demo_reply`. If the request fails, returns a message beginning with "Error:"."""
    if DEMO_MODE:
        return demo_reply
    try:
        response = client.models.generate_content(
            model = GEMMA_MODEL,
            contents = user_text,
            config = types.GenerateContentConfig(
                system_instruction = system_text,
                temperature = temperature,
                thinking_config = types.ThinkingConfig(thinking_level="minimal"),  # faster replies
            ),
        )
        return (response.text or "").strip()
    except errors.APIError as e:
        return _friendly_error(e)
    except Exception as e:
        return f"Error: {e}"


if DEMO_MODE:
    print("Demonstration mode: no GOOGLE_API_KEY was found. Retrieval will work; answers will show the passages found.")
else:
    print(f"Connected to Google AI Studio. Model: {GEMMA_MODEL}")
    print(generate("In one sentence, what is an open-weight AI model?"))

from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(EMBED_MODEL_NAME)   # runs on the CPU
print(f"Loaded embedding model: {EMBED_MODEL_NAME}")

## Step 3: Load the college documents

Downloads the prepared chunks and embeds each chunk's `embed_text`. Each chunk carries its document, page, section, heading and the office to contact.

To use a local copy instead (for example after re-running `scripts/prepare_dataset.py`), upload `chunks.jsonl` in the **Files** panel and set `CHUNKS_URL = "chunks.jsonl"` in Step 1.

In [ ]:
import json
import urllib.request
from collections import Counter

import numpy as np


def load_chunks(source=CHUNKS_URL):
    """Reads chunks.jsonl from a URL or a local path. Returns a list of chunk dictionaries."""
    if source.startswith("http"):
        with urllib.request.urlopen(source) as response:
            lines = response.read().decode("utf-8").splitlines()
    else:
        with open(source, encoding="utf-8") as f:
            lines = f.read().splitlines()
    return [json.loads(line) for line in lines if line.strip()]


knowledge_base = {"chunks": [], "vectors": None}  # filled by add_to_knowledge_base()


def add_to_knowledge_base(chunks):
    """Embeds each chunk's embed_text and adds the chunks to the knowledge base."""
    vectors = embedder.encode([c["embed_text"] for c in chunks], normalize_embeddings=True,
                              batch_size=64, show_progress_bar=True)
    knowledge_base["chunks"] += chunks
    knowledge_base["vectors"] = vectors if knowledge_base["vectors"] is None else np.vstack([knowledge_base["vectors"], vectors])


def is_ssit(c):
    """True for SSIT / SSAHE documents (and uploaded PDFs); False for the upGrad brochure."""
    return c.get("publisher", "SSIT / SSAHE") == "SSIT / SSAHE"


def course_label(c):
    """'B.E. CSE Year 3' for a syllabus chunk, None for the handbook."""
    return f"{c['program']} {c['branch']} Year {c['year']}" if c.get("type") == "syllabus" else None


def kb_summary():
    docs = Counter(c["doc"] for c in knowledge_base["chunks"])
    return f"Loaded **{len(docs)} documents**, **{len(knowledge_base['chunks'])} chunks**."


college_chunks = load_chunks()
if not INCLUDE_NON_SSIT:
    college_chunks = [c for c in college_chunks if is_ssit(c)]
add_to_knowledge_base(college_chunks)
COURSES = sorted({course_label(c) for c in college_chunks if course_label(c)})
print(kb_summary().replace("**", ""))
print("Courses:", ", ".join(COURSES))

## Step 4: The RAG pipeline

`retrieve()` finds the closest chunks (optionally only the student's course plus the handbook). `answer()` declines with the contact office when nothing is relevant, and otherwise asks Gemma to answer from the passages with document and page citations.

In [ ]:
from pypdf import PdfReader


def read_pdf(path):
    """Returns a list of page texts from a PDF file."""
    return [page.extract_text() or "" for page in PdfReader(path).pages]


def split_uploaded_pdf(path):
    """Splits an uploaded PDF into overlapping chunks of about CHUNK_SIZE words, in the same format as chunks.jsonl."""
    name = os.path.basename(path)
    chunks = []
    step = max(1, CHUNK_SIZE - CHUNK_OVERLAP)
    for page_number, text in enumerate(read_pdf(path), start=1):
        words = text.split()
        for start in range(0, len(words), step):
            chunk_words = words[start:start + CHUNK_SIZE]
            if len(chunk_words) > 5:  # skip very short fragments
                text_part = " ".join(chunk_words)
                chunks.append({"doc": name, "title": name, "type": "uploaded", "page": page_number,
                               "section": name, "heading": "", "contact": DEFAULT_CONTACT,
                               "text": text_part, "embed_text": f"{name} | {text_part}"})
            if start + CHUNK_SIZE >= len(words):
                break
    if not chunks:
        raise ValueError("No text was found. The PDF may contain scanned images only.")
    return chunks


def retrieve(question, top_k=TOP_K, course=None):
    """Finds the chunks whose meaning is closest to the question.
    With a course such as "B.E. CSE Year 3", only that course's syllabus chunks are searched, plus every non-syllabus chunk."""
    question_vector = embedder.encode([question], normalize_embeddings=True)[0]
    similarities = knowledge_base["vectors"] @ question_vector   # 1.0 = identical meaning
    if course and course != "All courses":
        allowed = np.array([course_label(c) in (None, course) for c in knowledge_base["chunks"]])
        similarities = np.where(allowed, similarities, -1.0)
    best = np.argsort(similarities)[::-1][:top_k]
    return [{**knowledge_base["chunks"][i], "score": float(similarities[i])} for i in best]


def source_label(c):
    label = f"{c['doc']}, Page {c['page']}" + (f", {c['heading'] or c['section']}" if (c.get('heading') or c.get('section')) else "")
    return label if is_ssit(c) else f"{label}] [NOT AN SSIT DOCUMENT - publisher: {c['publisher']}"


def answer(question, course=None):
    """Returns (reply, retrieved chunks). Declines with the contact office when no chunk is relevant enough."""
    found = retrieve(question, course=course)
    if not found or found[0]["score"] < MIN_SIMILARITY:
        return DECLINE.format(contact=DEFAULT_CONTACT), found

    context = "\n\n".join(f"[{source_label(c)}] (Contact: {c['contact'] or DEFAULT_CONTACT})\n{c['text']}" for c in found)
    prompt = f"Context:\n{context}\n\nQuestion: {question}"
    top = found[0]
    demo = (f"**Demonstration mode:** no API key is set, so Gemma did not write an answer. The most relevant passage is "
            f"from **{top['doc']}, p. {top['page']}** ({top['heading'] or top['section']}):\n\n> {' '.join(top['text'].split())[:400]}...")
    reply = generate(prompt, system_text=ANSWER_RULES, demo_reply=demo)
    return reply, found


# Quick test: one covered question and two that the documents do not cover.
for test_question in ["What is the minimum attendance required to write the semester end examination?",
                      "What are the hostel visiting hours?",
                      "Who won the 2022 FIFA World Cup?"]:
    reply, found = answer(test_question)
    print(f"\nQuestion: {test_question}\nAnswer: {reply}\nRetrieved passages:")
    for c in found:
        print(f"   [{c['doc']}, p. {c['page']}, similarity {c['score']:.2f}] {c['heading'][:70]}")

## Step 5: Launch the application

Run this cell to open the application below the cell and create a **public link** (`https://....gradio.live`). Re-run it after changing any cell above.

In [ ]:
import gradio as gr

gr.close_all()  # closes any previous version of the application that is still running

APP_TITLE = "SSIT Campus Regulations Assistant"
APP_DESCRIPTION = ("Ask about college rules (attendance, examinations, grading, promotion, activity points, scholarships, "
                   "anti-ragging, holidays) or your course syllabus. Answers come only from the official SSIT documents, "
                   "with the document and page for every fact.")


def on_upload(files):
    if not files:
        return kb_summary()
    try:
        for path in files:
            add_to_knowledge_base(split_uploaded_pdf(path))
        return kb_summary() + f" Added: {', '.join(os.path.basename(p) for p in files)}."
    except Exception as e:
        raise gr.Error(f"The PDF could not be read: {e}")


def on_ask(question, course):
    if not question.strip():
        raise gr.Error("Enter a question first.")
    reply, found = answer(question, course)
    sources = "\n\n".join(f"**{c['doc']}, p. {c['page']}** - {c['heading'] or c['section']} (similarity {c['score']:.2f})\n"
                          + ("" if is_ssit(c) else f"**Not an SSIT document** (publisher: {c['publisher']})\n")
                          + f"*Contact: {c['contact'] or DEFAULT_CONTACT}*\n> {' '.join(c['text'].split())}" for c in found)
    return reply, sources


with gr.Blocks(title=APP_TITLE) as demo:
    gr.Markdown(f"# {APP_TITLE}\n{APP_DESCRIPTION}")
    with gr.Row():
        course_box = gr.Dropdown(["All courses"] + COURSES, value="All courses", label="Your course (filters the syllabus; the handbook is always searched)")
        with gr.Column():
            status_md = gr.Markdown(kb_summary())
            pdf_input = gr.File(label="Add more PDFs (optional)", file_types=[".pdf"], file_count="multiple", type="filepath")
    question_box = gr.Textbox(label="Question", placeholder="For example: What is the minimum attendance to write the SEE?")
    ask_btn = gr.Button("Ask", variant="primary")
    gr.Examples([
        "What is the minimum attendance required to write the semester end examination?",
        "What is the minimum attendance for MBA students?",
        "How many AICTE activity points must a B.E. student earn?",
        "How many subjects can I carry over to the next year?",
        "What topics are in Unit 2 of Database Management System?",
        "What are the hostel visiting hours?",
        "Who won the 2022 FIFA World Cup?",
    ], inputs=question_box)
    answer_md = gr.Markdown()
    with gr.Accordion("Retrieved passages (the text given to Gemma)", open=False):
        sources_md = gr.Markdown()

    pdf_input.upload(on_upload, inputs=pdf_input, outputs=status_md)
    ask_btn.click(on_ask, inputs=[question_box, course_box], outputs=[answer_md, sources_md])
    question_box.submit(on_ask, inputs=[question_box, course_box], outputs=[answer_md, sources_md])

demo.launch(share=True)

## Next steps
1. **Test set:** write at least 15 questions with expected answers (include questions the documents do not cover), run them through `answer()`, and record the proportion answered correctly and the main failure cases.
2. **Tune `MIN_SIMILARITY`:** print the best score for covered and not-covered questions and pick a value between them.
3. **Experiments (bonus):** compare `TOP_K`, the embedding model, and `embed_text` versus `text` on the test set.
4. **RAG versus no RAG (bonus):** call `generate(question)` with no context and compare with `answer(question)`.

## Troubleshooting

| Issue | Resolution |
|---|---|
| The notebook remains in demonstration mode | Add `GOOGLE_API_KEY` in Secrets, enable **Notebook access**, and re-run the Step 2 cell. |
| "API key was rejected" | Create a new key at aistudio.google.com/apikey and update the secret. |
| "Usage limit reached" | Wait about one minute. Free keys allow a limited number of requests per minute. Team members may each create a key. |
| Step 3 cannot download the chunks | Check the internet connection, or upload `chunks.jsonl` in the Files panel and set `CHUNKS_URL = "chunks.jsonl"`. |
| `NameError: name '...' is not defined` | A cell was skipped. Run all cells above in order. |
| The public link no longer works | Links expire when the Colab session disconnects. Re-run the launch cell to create a new link. |